In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime

import sys
sys.path.append("../../")

# CCP Basis (CME-LCH) Fair Value Model

Hybrid multi-factor regression + swap curve PCA enrichment.

**Factors:**
- Swap curve PCA scores (PC1=level, PC2=slope, PC3=curvature)
- SOFR-EFFR spread (dealer funding cost)
- BGCR-EFFR spread (repo-OIS collateral cost)
- BBB Corporate OAS (dealer credit proxy)
- Realized rate volatility
- Rate level
- Quarter-end / year-end dummies

In [ ]:
from RVUtils.ccp_basis_fair_value import CCPBasisFairValueModel

In [ ]:
model = CCPBasisFairValueModel(
    tenors=["5Y", "10Y", "30Y"],
    start_date=datetime.date(2018, 6, 1),
    end_date=datetime.date(2026, 4, 1),
    n_pcs=3,
    realized_vol_window=20,
    ou_steps=63,
)

## 1. Fetch Raw Data

In [ ]:
raw_df = model.fetch_data()
print(f"Raw data shape: {raw_df.shape}")
print(f"Date range: {raw_df.index.min()} to {raw_df.index.max()}")
print(f"\nColumns: {raw_df.columns.tolist()}")
raw_df.tail()

## 2. Build Factor Matrix

In [ ]:
factors = model.build_factors()
print(f"Factor matrix shape: {factors.shape}")
print(f"\nFactors: {factors.columns.tolist()}")
factors.tail()

In [ ]:
# PCA variance explained
if model._pca_model is not None:
    ev = model._pca_model.eigenvalues
    var_explained = ev / ev.sum() * 100
    cum_var = var_explained.cumsum()
    print("PCA Variance Explained:")
    for pc in var_explained.index[:6]:
        print(f"  {pc}: {var_explained[pc]:.1f}% (cumulative: {cum_var[pc]:.1f}%)")
    
    # Plot PCA loadings
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, ax in enumerate(axes):
        pc = f"PC{i+1}"
        model._pca_model.loadings[pc].plot(kind='bar', ax=ax, title=f"{pc} Loadings ({var_explained[pc]:.1f}%)")
        ax.set_ylabel('Loading')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Fit All Tenors

In [ ]:
results = model.fit_all()
print(f"Successfully fitted: {list(results.keys())}")

In [ ]:
# Cross-tenor summary
summary = model.summary()
display(summary)

## 4. Fair Value Plots (per tenor)

In [ ]:
for tenor, result in results.items():
    model.plot_fair_value(result)
    plt.show()

## 5. Residual Z-Score Plots

In [ ]:
for tenor, result in results.items():
    model.plot_residual_zscore(result)
    plt.show()

## 6. Factor Attribution (30Y)

In [ ]:
if "30Y" in results:
    model.plot_factor_attribution(results["30Y"])
    plt.show()

## 7. Regression Coefficients Comparison

In [ ]:
coeff_df = model.coeff_table()
display(coeff_df.round(4))

In [ ]:
# Regression details for each tenor
for tenor, result in results.items():
    print(f"\n{'='*60}")
    print(f"  {tenor} Regression Summary")
    print(f"{'='*60}")
    print(result.regression_result.summary())

## 8. Basis Term Structure: Actual vs Fair Value

In [ ]:
model.plot_basis_term_structure()
plt.show()

## 9. OU Mean Reversion Analysis

In [ ]:
from RVUtils.plt_timeseries import make_secondary_axis_plot

for tenor, result in results.items():
    if result.ou_forecast is not None:
        print(f"\n{tenor}: OU half-life = {result.first_passage_time:.1f} days")
        
        plot, fig, ax, ax2, legend = make_secondary_axis_plot(
            ylabel_left="Residual (bps)",
            title=f"{tenor} Residual + OU Forecast",
        )
        
        # Last 252 days of residual + forecast
        plot(result.residual_ts.iloc[-252:], label="Residual")
        plot(result.ou_forecast["mean_reversion"], label="OU Mean Reversion")
        
        ax.fill_between(
            result.ou_forecast.index,
            result.ou_forecast["+1_sigma"],
            result.ou_forecast["-1_sigma"],
            alpha=0.15, color="blue", label="\u00b11\u03c3"
        )
        ax.fill_between(
            result.ou_forecast.index,
            result.ou_forecast["+2_sigma"],
            result.ou_forecast["-2_sigma"],
            alpha=0.08, color="blue", label="\u00b12\u03c3"
        )
        
        legend(show_date=True, loc="upper left")
        plt.show()

## 10. Factor Correlation Matrix

In [ ]:
import seaborn as sns

if "30Y" in results:
    corr = results["30Y"].factor_df.corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
    ax.set_title("Factor Correlation Matrix (30Y)")
    plt.tight_layout()
    plt.show()